# 5. Serve the model with FastAPI

This notebook loads the saved scikit-learn pipeline and exposes it through a small API. The API accepts the feature columns produced in notebook 2. It returns an anomaly percentile relative to the training scores.

In [ ]:
from pathlib import Path
import os
import uuid
import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict
from typing import Any

PROJECT = Path.cwd().resolve()
if not (PROJECT / "notebooks").exists():
    PROJECT = PROJECT.parent
ARTIFACT_DIR = Path(os.getenv("PPIP_ARTIFACTS_DIR", str(PROJECT / "artifacts")))
MODEL_PATH = ARTIFACT_DIR / "ppip_anomaly_model.joblib"
if not MODEL_PATH.exists():
    raise FileNotFoundError("Model not found. Run notebooks 1–4 before starting the API.")
BUNDLE = joblib.load(MODEL_PATH)
PIPELINE = BUNDLE["pipeline"]
REFERENCE_SCORES = np.asarray(BUNDLE["reference_scores"], dtype=float)
INPUT_COLUMNS = list(BUNDLE["input_columns"])

app = FastAPI(
    title="Kenya PPIP Procurement Screening API",
    description="Anomaly screening of procurement records for human review. Scores are not findings of wrongdoing.",
    version="1.0.0",
)

class ScoreRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    ocid: str | None = None
    features: dict[str, Any]

class BatchRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    records: list[ScoreRequest]


def score_record(request: ScoreRequest):
    extra = sorted(set(request.features) - set(INPUT_COLUMNS))
    if extra:
        raise HTTPException(status_code=422, detail=f"Unexpected feature names: {extra}")
    row = {
        column: request.features.get(column, "" if column == "tender_text" else np.nan)
        for column in INPUT_COLUMNS
    }
    if "tender_text" in row and row["tender_text"] is None:
        row["tender_text"] = ""
    for column in BUNDLE.get("numeric_columns", []):
        row[column] = pd.to_numeric(pd.Series([row[column]]), errors="coerce").iloc[0]
    for column in BUNDLE.get("categorical_columns", []):
        if row[column] is None:
            row[column] = np.nan
    frame = pd.DataFrame([row], columns=INPUT_COLUMNS)
    raw_score = float(-PIPELINE.decision_function(frame)[0])
    percentile = float(np.searchsorted(REFERENCE_SCORES, raw_score, side="right") / len(REFERENCE_SCORES))
    band = "high" if percentile >= 0.98 else "review" if percentile >= 0.90 else "routine"
    return {
        "request_id": str(uuid.uuid4()),
        "ocid": request.ocid,
        "anomaly_percentile": round(percentile, 4),
        "review_band": band,
        "score_type": "relative anomaly percentile",
        "notice": "This score is a screening signal only. It is not a probability or proof of fraud, corruption or collusion.",
    }

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/model-info")
def model_info():
    return {
        "model": "Isolation Forest",
        "version": BUNDLE.get("model_version", "1.0"),
        "training_start": BUNDLE.get("train_start"),
        "training_end": BUNDLE.get("train_end"),
        "feature_count": len(INPUT_COLUMNS),
        "source_sha256": BUNDLE.get("source_sha256"),
        "verified_labels_available": False,
    }

@app.post("/score")
def score(request: ScoreRequest):
    return score_record(request)

@app.post("/score/batch")
def score_batch(request: BatchRequest):
    if len(request.records) > 100:
        raise HTTPException(status_code=422, detail="A batch can contain at most 100 records.")
    return {"results": [score_record(record) for record in request.records]}

## Test the API

Run these checks after the model has been trained. The tests use one real feature row from the holdout table to check the response shape and the health endpoint.

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)
assert client.get("/health").status_code == 200
assert client.get("/model-info").status_code == 200

sample_file = PROJECT / "data" / "processed" / "ppip_features.csv"
if sample_file.exists():
    sample = pd.read_csv(sample_file).iloc[0]
    payload = {}
    for column in INPUT_COLUMNS:
        value = sample[column] if column in sample.index else None
        if pd.isna(value):
            payload[column] = None
        elif hasattr(value, "item"):
            payload[column] = value.item()
        else:
            payload[column] = value
    response = client.post("/score", json={"ocid": str(sample["ocid"]), "features": payload})
    assert response.status_code == 200, response.text
    result = response.json()
    assert 0.0 <= result["anomaly_percentile"] <= 1.0
    assert result["review_band"] in {"routine", "review", "high"}
    print("API checks passed.")
    print(result)
else:
    print("Health and model-info checks passed. Run notebooks 1–4 to test /score with a prepared record.")


## Start the service

The service is started by `run_api.sh` or Docker. The final cell is left idle during notebook review unless `START_API=1` is set in the environment.

In [ ]:
if os.getenv("START_API") == "1":
    import uvicorn
    uvicorn.run(app, host=os.getenv("API_HOST", "0.0.0.0"), port=int(os.getenv("API_PORT", "8000")))
else:
    print("API is defined and tested. Set START_API=1 to start the server.")